# Customer Lifetime Value (CLV) Prediction using Machine Learning

**A Beginner-to-Intermediate Data Science Project**

This notebook builds an end-to-end machine learning pipeline to predict Customer Lifetime Value (CLV) using simple, transparent techniques — specifically, Linear Regression on features engineered from raw transaction data.

---


## 1. Business Problem

### What is Customer Lifetime Value (CLV)?
Customer Lifetime Value (CLV) is an estimate of the **total revenue a business can expect from a single customer** over the course of their relationship with that business. In simple terms: *"How much money will this customer likely spend with us in the future?"*

### Why do businesses predict CLV?
- **Resource allocation:** Businesses have limited marketing budgets. Knowing which customers are likely to be most valuable helps them spend money where it matters most.
- **Customer prioritization:** Not all customers are equally valuable. A business can treat high-CLV customers differently (e.g., loyalty perks, faster support) than low-CLV customers.
- **Revenue forecasting:** Predicting CLV across the customer base helps businesses forecast future revenue more accurately.
- **Acquisition strategy:** If a business knows the CLV of customers who came from a certain marketing channel, it can decide how much it's worth spending to acquire similar customers.

### How does CLV help in customer retention and marketing?
- **Retention:** By identifying customers with high predicted CLV but declining activity, a business can proactively reach out with offers to retain them before they churn.
- **Personalized marketing:** Customers can be segmented into groups (e.g., high, medium, low CLV) and given personalized offers, emails, or discounts suited to their value and behavior.
- **Customer acquisition cost (CAC) decisions:** If acquiring a customer costs more than their predicted CLV, that acquisition channel may not be profitable — CLV helps set sensible spending limits.

In short, CLV turns raw transaction history into a forward-looking business signal that guides marketing, retention, and customer service decisions.


## 2. Dataset Selection

### Dataset: Online Retail Dataset (Kaggle)
**Source:** [Kaggle — E-Commerce Data by carrie1](https://www.kaggle.com/datasets/carrie1/ecommerce-data)
*(Originally from the UCI Machine Learning Repository — "Online Retail Dataset")*

### Why this dataset?
- **Widely used and well-documented:** It is one of the most popular datasets for customer analytics, RFM (Recency-Frequency-Monetary) analysis, and CLV projects — meaning there is a strong community of tutorials and prior work to learn from.
- **Real transaction-level data:** It contains ~541,909 individual transaction line items from a UK-based online retail store between **01 Dec 2010 and 09 Dec 2011**, which is exactly the kind of raw data needed to engineer customer-level features (Total Spend, Purchase Frequency, etc.) from scratch.
- **Sufficient size:** With over half a million transaction rows and thousands of unique customers, it's large enough to train and evaluate a regression model meaningfully, but not so large that it requires heavy infrastructure.
- **Beginner-friendly structure:** The dataset has a simple, flat structure (`InvoiceNo`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `UnitPrice`, `CustomerID`, `Country`) — no complex joins or relational schemas needed.
- **Naturally supports a CLV framing:** Because it spans about a year of transactions, we can split the timeline into a "calibration period" (used to build features about past behavior) and a "holdout period" (used to measure actual future spend — our CLV target). This lets us build a genuine **predictive** CLV model without needing time-series forecasting techniques.

**Note:** Download the dataset from Kaggle (link above) or let the notebook download it automatically via `kagglehub` in the next section. The file is typically named `data.csv` (encoded in `ISO-8859-1`).


## 3. Import Libraries

In [ ]:
# Core libraries for data handling
import pandas as pd
import numpy as np

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing and model selection
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Model
from sklearn.linear_model import LinearRegression

# Evaluation metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make plots look nicer
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

# For reproducibility
RANDOM_STATE = 42


## 4. Load the Dataset

We'll download the dataset directly from Kaggle using `kagglehub`. This requires a one-time Kaggle API setup:

1. Create a free Kaggle account: https://www.kaggle.com
2. Go to **Kaggle → Settings → API → Create New Token** to download `kaggle.json`.
3. Place it at `~/.kaggle/kaggle.json` (Mac/Linux) or `C:\Users\<you>\.kaggle\kaggle.json` (Windows), or set the `KAGGLE_USERNAME` / `KAGGLE_KEY` environment variables.

> **Alternative (manual) method:** Download `data.csv` manually from https://www.kaggle.com/datasets/carrie1/ecommerce-data, place it in the same folder as this notebook, and load it directly with:
> ```python
> df = pd.read_csv("data.csv", encoding="ISO-8859-1")
> ```


In [ ]:
# Install kagglehub if not already installed (only needs to run once)
# Uncomment the line below if kagglehub is not installed in your environment
# !pip install kagglehub

import kagglehub
import os

# Download the dataset directly from Kaggle
dataset_path = kagglehub.dataset_download("carrie1/ecommerce-data")

print("Dataset downloaded to:", dataset_path)
print("Files in dataset folder:", os.listdir(dataset_path))

# Build the full path to the CSV file and load it with Pandas
# The file uses ISO-8859-1 encoding (it contains special characters)
csv_path = os.path.join(dataset_path, "data.csv")
df = pd.read_csv(csv_path, encoding="ISO-8859-1")

# Look at the first few rows
df.head()


In [ ]:
# Basic information about the dataset
print("Shape of the dataset:", df.shape)
print("\nColumn data types:")
df.info()


## 5. Data Cleaning

Real-world transaction data is messy. We need to:
- Check for missing values.
- Handle duplicate rows.
- Convert data types where required (e.g., `InvoiceDate` to a proper datetime).
- Remove invalid transactions (e.g., cancellations, negative quantities/prices).


In [ ]:
# Check for missing values in each column
missing_values = df.isnull().sum()
print("Missing values per column:")
print(missing_values[missing_values > 0])


**Observation:** The `CustomerID` column has a large number of missing values. Since our entire project is about predicting *customer*-level CLV, transactions without a `CustomerID` cannot be attributed to any customer and must be dropped. `Description` also has a few missing values, but since we don't use it for modeling, we can leave those rows (they'll simply be excluded once we drop missing `CustomerID` rows, in most cases).

In [ ]:
# Drop rows with missing CustomerID, since we can't build customer-level features without it
df = df.dropna(subset=["CustomerID"])

# Convert CustomerID to integer type (it's currently a float due to missing values)
df["CustomerID"] = df["CustomerID"].astype(int)

print("Shape after dropping missing CustomerID rows:", df.shape)


In [ ]:
# Check for duplicate rows
duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_count}")

# Remove duplicates if any exist
if duplicate_count > 0:
    df = df.drop_duplicates()
    print(f"Duplicates removed. New shape: {df.shape}")


In [ ]:
# Convert InvoiceDate to a proper datetime type
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

print("InvoiceDate range:", df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())


**Removing invalid transactions:**
- Invoice numbers starting with `"C"` represent **cancelled orders**. These should be excluded since they don't represent genuine revenue.
- `Quantity` or `UnitPrice` values that are zero or negative don't represent valid sales (they're often returns, data entry errors, or adjustments) and should be removed.

In [ ]:
# Remove cancelled orders (InvoiceNo starting with 'C')
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]

# Keep only rows with positive Quantity and positive UnitPrice
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]

print("Shape after removing cancellations and invalid rows:", df.shape)


In [ ]:
# Create a 'TotalPrice' column: the revenue generated by each transaction line
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]

df.head()


## 6. Exploratory Data Analysis (EDA)

Let's explore customer spending patterns, purchase frequency, and revenue distribution before building our model.

### 6.1 Customer-Level Aggregation (for EDA)

In [ ]:
# Aggregate transaction data to customer level for EDA purposes
customer_summary = df.groupby("CustomerID").agg(
    TotalSpend=("TotalPrice", "sum"),
    NumPurchases=("InvoiceNo", "nunique"),
    AvgPurchaseValue=("TotalPrice", "mean")
).reset_index()

customer_summary.head()


### 6.2 Distribution of Customer Spending

In [ ]:
# Distribution of total spend per customer (zoomed in to remove extreme outliers for visibility)
plt.figure(figsize=(8, 5))
sns.histplot(customer_summary["TotalSpend"], bins=60, kde=True, color="teal")
plt.title("Distribution of Total Customer Spending")
plt.xlabel("Total Spend")
plt.ylabel("Number of Customers")
plt.xlim(0, customer_summary["TotalSpend"].quantile(0.95))  # zoom to 95th percentile
plt.show()


**Observation:** Customer spending is heavily right-skewed — most customers spend a relatively small amount, while a small number of customers spend very large amounts. This is typical in retail and is an important pattern for CLV modeling and customer segmentation.

### 6.3 Purchase Frequency

In [ ]:
# Distribution of number of purchases (invoices) per customer
plt.figure(figsize=(8, 5))
sns.histplot(customer_summary["NumPurchases"], bins=40, kde=False, color="darkorange")
plt.title("Distribution of Purchase Frequency (Number of Orders per Customer)")
plt.xlabel("Number of Purchases")
plt.ylabel("Number of Customers")
plt.xlim(0, customer_summary["NumPurchases"].quantile(0.95))
plt.show()


**Observation:** Most customers place only a handful of orders, while a small group of loyal/frequent customers places many more. This "few high-frequency customers" pattern often overlaps with high-spending customers.

### 6.4 Revenue Distribution (by Country)

In [ ]:
# Total revenue by country (top 10)
revenue_by_country = df.groupby("Country")["TotalPrice"].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=revenue_by_country.values, y=revenue_by_country.index, palette="viridis")
plt.title("Top 10 Countries by Total Revenue")
plt.xlabel("Total Revenue")
plt.ylabel("Country")
plt.show()


**Observation:** The vast majority of revenue comes from the United Kingdom, since this is a UK-based online retailer. This tells us `Country` may still carry some signal (UK vs non-UK), but won't be a rich feature on its own.

### 6.5 Correlation Heatmap

In [ ]:
# Correlation heatmap of customer-level numeric features
plt.figure(figsize=(6, 5))
sns.heatmap(customer_summary[["TotalSpend", "NumPurchases", "AvgPurchaseValue"]].corr(),
            annot=True, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap of Customer-Level Features")
plt.show()


### 6.6 Histograms of Important Features

In [ ]:
# Histograms of key customer-level features
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.histplot(customer_summary["TotalSpend"], bins=50, ax=axes[0], color="teal")
axes[0].set_title("Total Spend")
axes[0].set_xlim(0, customer_summary["TotalSpend"].quantile(0.95))

sns.histplot(customer_summary["NumPurchases"], bins=30, ax=axes[1], color="darkorange")
axes[1].set_title("Number of Purchases")
axes[1].set_xlim(0, customer_summary["NumPurchases"].quantile(0.95))

sns.histplot(customer_summary["AvgPurchaseValue"], bins=50, ax=axes[2], color="purple")
axes[2].set_title("Average Purchase Value")
axes[2].set_xlim(0, customer_summary["AvgPurchaseValue"].quantile(0.95))

plt.tight_layout()
plt.show()


### 6.7 Boxplots to Detect Outliers

In [ ]:
# Boxplots help us see the spread and outliers in spending behavior
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(y=customer_summary["TotalSpend"], ax=axes[0], color="skyblue")
axes[0].set_title("Boxplot: Total Spend")
axes[0].set_ylim(0, customer_summary["TotalSpend"].quantile(0.95))

sns.boxplot(y=customer_summary["NumPurchases"], ax=axes[1], color="salmon")
axes[1].set_title("Boxplot: Number of Purchases")
axes[1].set_ylim(0, customer_summary["NumPurchases"].quantile(0.95))

plt.tight_layout()
plt.show()


### 6.8 Scatter Plot Between Important Variables

In [ ]:
# Scatter plot: Number of Purchases vs Total Spend
plt.figure(figsize=(8, 5))
sns.scatterplot(x="NumPurchases", y="TotalSpend", data=customer_summary, alpha=0.4, color="teal")
plt.title("Number of Purchases vs Total Spend")
plt.xlabel("Number of Purchases")
plt.ylabel("Total Spend")
plt.xlim(0, customer_summary["NumPurchases"].quantile(0.98))
plt.ylim(0, customer_summary["TotalSpend"].quantile(0.98))
plt.show()


**Observation:** There's a clear positive relationship — customers who purchase more often also tend to spend more in total. This makes intuitive sense and confirms that purchase frequency will likely be a useful predictor of CLV.

## 7. Feature Engineering

To predict CLV, we frame this as: **"Given a customer's behavior in an initial time period (calibration period), can we predict how much they will spend in a later time period (holdout period)?"**

This gives us a genuine *predictive* task (not just describing the past) while staying simple — no time-series forecasting models are needed, just a smart split of the data by date and simple aggregation, followed by regression.

### Splitting the timeline
- **Calibration period:** the first 9 months of data — used to build features describing past customer behavior.
- **Holdout period:** the last 3 months of data — the actual amount each customer spent here becomes our CLV **target** (the value we want to predict).


In [ ]:
# Define the cutoff date to split calibration and holdout periods
max_date = df["InvoiceDate"].max()
min_date = df["InvoiceDate"].min()
print("Data ranges from", min_date, "to", max_date)

# Use the last 3 months as the holdout period, everything before that as calibration
cutoff_date = max_date - pd.DateOffset(months=3)
print("Cutoff date:", cutoff_date)

calibration_df = df[df["InvoiceDate"] <= cutoff_date]
holdout_df = df[df["InvoiceDate"] > cutoff_date]

print("Calibration period rows:", calibration_df.shape[0])
print("Holdout period rows:", holdout_df.shape[0])


### Engineering features from the calibration period

We create the following simple, easy-to-understand features for each customer, based only on their calibration-period behavior:

- **Total Spend:** The total amount a customer has spent so far. A direct signal of past value.
- **Number of Purchases:** How many separate orders (invoices) the customer has placed. Reflects engagement/loyalty.
- **Average Purchase Value:** Total Spend divided by Number of Purchases — reflects whether a customer buys cheap items frequently or expensive items occasionally.
- **Customer Tenure:** Number of days between a customer's first purchase and the cutoff date. Longer tenure often means a more established, loyal customer.
- **Purchase Frequency:** Number of Purchases divided by Tenure (in days) — how often, on average, the customer buys (purchases per day). Captures buying "pace" independent of how long they've been a customer.
- **Recency:** Number of days between the customer's most recent purchase and the cutoff date. A customer who purchased recently is more likely to be active and engaged than one who hasn't purchased in a long time.


In [ ]:
# Build calibration-period features per customer
features = calibration_df.groupby("CustomerID").agg(
    TotalSpend=("TotalPrice", "sum"),
    NumPurchases=("InvoiceNo", "nunique"),
    FirstPurchaseDate=("InvoiceDate", "min"),
    LastPurchaseDate=("InvoiceDate", "max"),
    Country=("Country", "first")
).reset_index()

# Average Purchase Value = Total Spend / Number of Purchases
features["AvgPurchaseValue"] = features["TotalSpend"] / features["NumPurchases"]

# Customer Tenure = days between first purchase and the cutoff date
features["Tenure"] = (cutoff_date - features["FirstPurchaseDate"]).dt.days

# Recency = days between last purchase and the cutoff date (smaller = more recently active)
features["Recency"] = (cutoff_date - features["LastPurchaseDate"]).dt.days

# Purchase Frequency = purchases per day since first purchase (avoid divide-by-zero with +1)
features["PurchaseFrequency"] = features["NumPurchases"] / (features["Tenure"] + 1)

features.head()


### Building the target: future spend (CLV) in the holdout period

In [ ]:
# Calculate how much each customer actually spent in the holdout period
# This is our target variable: the "future CLV" we want to predict
holdout_spend = holdout_df.groupby("CustomerID")["TotalPrice"].sum().reset_index()
holdout_spend.columns = ["CustomerID", "FutureSpend"]

holdout_spend.head()


In [ ]:
# Merge features (calibration period) with the target (holdout period spend)
# We use a LEFT JOIN on calibration customers: customers who didn't purchase again in the
# holdout period simply get a FutureSpend of 0 (this is a realistic, valid outcome to predict)
data = features.merge(holdout_spend, on="CustomerID", how="left")
data["FutureSpend"] = data["FutureSpend"].fillna(0)

print("Final dataset shape:", data.shape)
data.head()


## 8. Data Preprocessing

Now we prepare the data for modeling:
- Select the useful features.
- Encode the categorical `Country` variable.
- Scale numerical features (important for interpreting Linear Regression coefficients fairly).
- Split into training and testing sets.


In [ ]:
# Simplify Country into a binary feature: is the customer from the UK or not?
# (The UK dominates the dataset, so a simple binary flag is enough — no need for complex encoding)
data["IsUK"] = (data["Country"] == "United Kingdom").astype(int)

# Select the features we'll use for modeling
feature_columns = ["TotalSpend", "NumPurchases", "AvgPurchaseValue", "Tenure", "Recency", "PurchaseFrequency", "IsUK"]

X = data[feature_columns]
y = data["FutureSpend"]

print("Feature set shape:", X.shape)
print("Target shape:", y.shape)


In [ ]:
# Split into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)


In [ ]:
# Scale numerical features using StandardScaler
# Scaling puts all features on the same scale, which also makes Linear Regression
# coefficients easier to compare later (to judge feature importance)
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete.")


## 9. Train the Linear Regression Model

Linear Regression is a simple, interpretable model that predicts a numeric target as a weighted sum of the input features. It's a great starting point for CLV prediction because it's easy to explain to business stakeholders — each feature gets a coefficient showing its impact on predicted spend.

In [ ]:
# Train the Linear Regression model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test_scaled)

print("Model trained successfully.")


## 10. Model Evaluation

We evaluate the model using four standard regression metrics.

In [ ]:
# Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE) : {mae:.2f}")
print(f"Mean Squared Error (MSE)  : {mse:.2f}")
print(f"Root Mean Squared Error   : {rmse:.2f}")
print(f"R² Score                  : {r2:.4f}")


### 10.1 Predicted vs Actual Spend

In [ ]:
# Scatter plot of predicted vs actual future spend
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.4, color="teal")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color="red", linestyle="--", label="Perfect Prediction")
plt.xlabel("Actual Future Spend")
plt.ylabel("Predicted Future Spend")
plt.title("Predicted vs Actual Future Spend (CLV)")
plt.xlim(0, y_test.quantile(0.98))
plt.ylim(0, y_test.quantile(0.98))
plt.legend()
plt.show()


## 11. Understanding the Evaluation Metrics (In Simple Terms)

- **Mean Absolute Error (MAE):** The average size of our prediction errors, in the same units as the target (currency). If MAE is, say, 150, it means on average our predictions are off by about 150 (in currency) from the true future spend — regardless of whether we overestimated or underestimated.

- **Mean Squared Error (MSE):** Similar to MAE, but it squares each error before averaging. This means **larger errors are penalized much more heavily** than small ones. MSE is harder to interpret directly (its units are "currency squared"), but it's useful for comparing models.

- **Root Mean Squared Error (RMSE):** The square root of MSE, which brings the units back to the original scale (currency). RMSE is generally a bit larger than MAE when there are some large errors (outliers) in the predictions, since it punishes big mistakes more.

- **R² Score (Coefficient of Determination):** Measures how much of the variation in actual future spend is "explained" by our model, on a scale from 0 to 1 (it can also go negative for a very poor model). An R² of 0.65, for example, means the model explains about 65% of the variation in customer future spend — the remaining 35% is due to factors the model doesn't capture.

### Interpreting our model's performance
Compare the MAE and RMSE to the average value of `FutureSpend` in the dataset (printed below) to judge whether the error is "small" or "large" in a practical, business sense — an error of 100 means very different things if average spend is 50 versus if it's 5,000.

In [ ]:
# Compare error metrics to the average future spend, for context
print(f"Average actual future spend in test set: {y_test.mean():.2f}")
print(f"MAE as a percentage of average spend: {(mae / y_test.mean()) * 100:.1f}%")


## 12. Most Influential Features (Linear Regression Coefficients)

Because we scaled all features before training, we can directly compare the magnitude of each coefficient to understand which features have the strongest impact on predicted CLV. A larger positive coefficient means that feature increases predicted future spend; a larger negative coefficient means it decreases predicted future spend.

In [ ]:
# Build a dataframe of feature coefficients, sorted by absolute impact
coefficients = pd.DataFrame({
    "Feature": feature_columns,
    "Coefficient": model.coef_
})
coefficients["AbsCoefficient"] = coefficients["Coefficient"].abs()
coefficients = coefficients.sort_values(by="AbsCoefficient", ascending=False)

coefficients


In [ ]:
# Visualize feature importance (coefficient magnitude)
plt.figure(figsize=(8, 5))
sns.barplot(x="Coefficient", y="Feature", data=coefficients, palette="viridis")
plt.title("Linear Regression Coefficients (Feature Importance for Predicting CLV)")
plt.xlabel("Coefficient (impact on predicted future spend)")
plt.axvline(0, color="black", linewidth=0.8)
plt.show()


**Interpretation:** Features with large positive coefficients (commonly `TotalSpend`, `NumPurchases`, and `PurchaseFrequency` in this kind of dataset) suggest that customers who have spent more, purchased more often, or purchase more frequently in the calibration period are predicted to spend more in the future. A large negative coefficient on `Recency` would suggest that customers who haven't purchased recently are predicted to spend less going forward — an intuitive and business-relevant insight.

*(Exact ranking will depend on your specific run — check the printed table above and update this interpretation with your actual top features.)*

## 13. Business Recommendations

Based on this CLV prediction model, here are practical ways a business could use these insights:

1. **Identify high-value customers.** Use the model's predicted future spend to rank customers, and flag the top 10-20% as "high-value" — these customers deserve prioritized customer service, exclusive offers, or account management attention.

2. **Segment customers based on predicted CLV.** Group customers into tiers (e.g., High / Medium / Low predicted CLV) using simple thresholds (like quartiles) on the predicted values. This turns a continuous prediction into an actionable segmentation the marketing team can use directly.

3. **Personalize marketing strategies.**
   - **High predicted CLV + high Recency (haven't purchased in a while):** These are valuable customers at risk of churning — target them with win-back campaigns or personalized discounts.
   - **High predicted CLV + low Recency (recently active):** Reward loyalty with premium offers, early access to new products, or loyalty points to reinforce the relationship.
   - **Low predicted CLV customers:** Focus on lower-cost marketing channels (e.g., automated email campaigns) rather than expensive personalized outreach.

4. **Customer retention opportunities.** Since `Recency` and `PurchaseFrequency` are likely strong predictors, the business can set up automatic alerts when a historically valuable customer's recency starts increasing (i.e., they haven't purchased in a while), triggering a proactive retention campaign before the customer is lost entirely.

5. **Inform customer acquisition spend.** If the business knows the average predicted CLV of customers acquired through a certain channel or campaign, it can set a sensible cap on how much to spend acquiring similar customers (Customer Acquisition Cost should stay well below predicted CLV).

---

## 14. Project Summary

In this project, we:
- Framed Customer Lifetime Value prediction as a business problem and explained why it matters.
- Selected and justified a real-world Kaggle transaction dataset suited for beginner-level CLV modeling.
- Cleaned messy, real-world transactional data (missing IDs, cancellations, invalid rows).
- Performed EDA to understand customer spending, frequency, and revenue patterns.
- Engineered simple, explainable features (Total Spend, Purchases, Tenure, Recency, Purchase Frequency) using a calibration/holdout time split — avoiding both circular targets and complex time-series forecasting.
- Trained an interpretable Linear Regression model and evaluated it with MAE, MSE, RMSE, and R².
- Identified the most influential features using model coefficients.
- Translated model results into concrete business recommendations for retention, segmentation, and marketing.

This project demonstrates a complete, beginner-friendly, end-to-end data science workflow for a genuinely useful business problem — from framing to feature engineering, modeling, evaluation, and business impact.
